# Group cells by their neighbourhood

The first notebook grouped cells by **their own shape**. Here we group them by
**the shapes around them**. Two cells can look different and still sit in the
same kind of neighbourhood, and the other way round. We call a recurring kind
of neighbourhood a **niche**.

How it works:

1. Every cell has a set of shape numbers (its principal components, PC1...).
2. For each cell we average those numbers with the ones of the cells it touches. We repeat this a few times, so the average reaches cells further away ("hops"). A cell that touches no other cell has no neighbourhood and is left out of the niche analysis.
3. The averaged numbers describe the neighbourhood. We map them (UMAP) and cluster them, just like in the first notebook.
4. We then ask which shape clusters live in which niches.

**You need:**

- `shape_embedding.csv` from the first notebook.
- One contact file per stack, named `<stack_name>_adjacency.csv`, with the columns `label_a`, `label_b`, `contact_voxels`. The segmentation tutorial writes them next to its feature table. The plugin's Step 4 writes them to `artifacts/step4/qc/` of the workspace, when neighbour features were measured; its Export window writes only the merged table, not the contact files.
- Every cell of the stacks must still be in the shape table: do not use a table made with a cell subset, because a cell's neighbours that are missing from the table cannot enter its neighbourhood.

**You get**, in the output folder: `niche_embedding.csv` (the shape table plus the niche columns), `niche_analysis_config.json`, one `representative_cells.csv` per setting, and the figures.

In [ ]:
# Make the helper file next to this notebook importable (works from the repo root or from this folder).
import sys
from pathlib import Path

for _p in (Path.cwd(), Path.cwd() / "tutorials" / "analysis", Path.cwd().parent):
    if (_p / "_helpers.py").exists():
        sys.path.insert(0, str(_p))
        break
else:
    raise FileNotFoundError("Could not find _helpers.py. Open this notebook from the repository root or from tutorials/analysis.")

import json
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import _helpers as h

warnings.filterwarnings("ignore", category=FutureWarning)
print("Ready.")

## 1. Edit this cell

- `SHAPE_EMBEDDING_CSV`: the file written by the first notebook.
- `ADJACENCY_DIR`: the folder with the `<stack_name>_adjacency.csv` files. `None` searches, in this order: the folder recorded by the first notebook, the folder of `SHAPE_EMBEDDING_CSV`, the folder above it, and `artifacts/step4/qc/` of a plugin workspace above it. If nothing is found, the notebook lists the folders it tried.
- `OUTPUT_DIR`: where the results are written.
- `NICHE_CONFIGS`: the neighbourhood recipes to compare. `n_iter` is the number of hops. `self_weight` is how much of the cell's own value is added back at every step. With `self_weight` 0 the cell's own value is not added back; after two or more hops it still returns a little through its neighbours.
- `PRIMARY_CONFIG`: the recipe whose result is saved in `niche_embedding.csv` and used for the shape-by-niche table. `hop3_pure` (three hops, own value not added back) is a reasonable start: three hops reach the next two rings of cells around a cell.
- `PRIMARY_NICHE_RES`: the niche resolution used for the plots and the shape-by-niche table. It must be one of `NICHE_LEIDEN_RES`.
- `SHAPE_COL`: the shape cluster column to compare the niches with. `None` uses the resolution the first notebook marked as active.
- `MIN_CONTACT`: contacts smaller than this are ignored (see the note under the next heading). `WEIGHT_BY_CONTACT`: `True` lets a bigger contact count more.
- `NICHE_LEIDEN_RES`: the clustering resolutions for the niche clusters (see how to choose a resolution in the first notebook).
- `KNN`, `UMAP_NEIGHBORS`, `UMAP_MIN_DIST`, `RANDOM_SEED`, `N_REPS`, `SAVE_FIGURES`: defaults that work for most tables.
- `MAX_CELLS_FOR_MAP`: for very large tables, draw each niche map from a random subset of this many cells. The averaging and the clusters always use every cell with a neighbourhood. `None` maps every cell. Five maps of a few hundred thousand cells take a long time; set a number if you wait too long.
- `LABELS_TIFF`, `MESHES_DIR`: optional, to draw typical cells, as in the first notebook.

In [ ]:
# ---------------- EDIT ME ----------------
SHAPE_EMBEDDING_CSV = "shape_output/shape_embedding.csv"
ADJACENCY_DIR = None
OUTPUT_DIR = "niche_output"

NICHE_CONFIGS = [
    {"name": "hop1_w0.5", "self_weight": 0.5, "n_iter": 1},
    {"name": "hop3_w0.5", "self_weight": 0.5, "n_iter": 3},
    {"name": "hop5_w0.5", "self_weight": 0.5, "n_iter": 5},
    {"name": "hop1_pure", "self_weight": 0.0, "n_iter": 1},
    {"name": "hop3_pure", "self_weight": 0.0, "n_iter": 3},
]
PRIMARY_CONFIG = "hop3_pure"
PRIMARY_NICHE_RES = 1.5         # niche resolution used for the plots and the shape-by-niche table
SHAPE_COL = None                # which shape cluster column to compare with; None = the first notebook's active resolution

MIN_CONTACT = 100               # ignore contacts smaller than this many voxels
WEIGHT_BY_CONTACT = True        # True = a bigger contact counts more

LABELS_TIFF = None
LABELS_STACK_NAME = None
VOXEL_SIZE_ZYX = (1.0, 1.0, 1.0)
MESHES_DIR = None

# ---------------- Settings with defaults ----------------
NICHE_LEIDEN_RES = [0.5, 1.0, 1.5]   # clustering resolutions for the niche clusters
KNN = 15
MAX_CELLS_FOR_MAP = None             # None = map every cell; a number = draw each niche map from a random subset
UMAP_NEIGHBORS = 15
UMAP_MIN_DIST = 0.1
RANDOM_SEED = 42
N_REPS = 10
SAVE_FIGURES = True

## 2. Load the shape table and find the contact files

The notebook takes the principal-component columns (`PC1`, `PC2`...) from the shape table and checks that every stack has a contact file.

In [ ]:
emb_path = Path(SHAPE_EMBEDDING_CSV)
out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)
fig_dir = out_dir / "figures"

emb = h.read_table_csv(emb_path)
pc_cols = sorted([c for c in emb.columns if re.fullmatch(r"PC\d+", c)], key=lambda c: int(c[2:]))
if not pc_cols:
    raise ValueError("No PC1, PC2... columns found. Run the first notebook and use its shape_embedding.csv.")
pcs = emb[pc_cols].to_numpy(dtype=float)
print(f"{len(emb):,} cells, {emb['stack_name'].nunique()} stack(s), {len(pc_cols)} principal components.")

# Settings the first notebook recorded next to its table.
cfg_path = emb_path.parent / "analysis_config.json"
cfg1 = json.loads(cfg_path.read_text(encoding="utf-8")) if cfg_path.exists() else {}
if cfg1.get("max_cells") is not None:
    raise ValueError("This shape table was made from a random subset of cells (MAX_CELLS). Neighbourhoods need every cell: "
                     "run the first notebook again without a cell limit.")

adj_dir = h.find_adjacency_dir(emb["stack_name"].unique(), emb_path, ADJACENCY_DIR, cfg1.get("adjacency_dir"))
have = [s for s in emb["stack_name"].unique() if (adj_dir / f"{s}_adjacency.csv").exists()]
print(f"Contact files: {len(have)} of {emb['stack_name'].nunique()} stacks found in {adj_dir}")
if SHAPE_COL is None:
    active = cfg1.get("active_resolution")
    lc = [c for c in emb.columns if c.startswith("leiden_r")]
    SHAPE_COL = h.res_name(active) if active is not None and h.res_name(active) in lc else (
        "leiden_r1.0" if "leiden_r1.0" in lc else (lc[0] if lc else None))
print("Shape cluster column:", SHAPE_COL)

if float(PRIMARY_NICHE_RES) not in [float(r) for r in NICHE_LEIDEN_RES]:
    raise ValueError(f"PRIMARY_NICHE_RES={PRIMARY_NICHE_RES} must be one of NICHE_LEIDEN_RES={NICHE_LEIDEN_RES}.")
if PRIMARY_CONFIG not in [c["name"] for c in NICHE_CONFIGS]:
    raise ValueError(f"PRIMARY_CONFIG '{PRIMARY_CONFIG}' is not one of {[c['name'] for c in NICHE_CONFIGS]}")

### Choosing `MIN_CONTACT`

`contact_voxels` measures how much two cells touch. It is counted in voxels in
a thin shell around the cell, so it grows with the size of the touching surface.
What counts as "a real contact" depends on your voxel size and cell size. Very
small contacts are often only corners where cells meet.

The plot shows all contacts of the first stack (log scale) and the line at
`MIN_CONTACT`. A good value sits in the valley between many tiny contacts and
the bulk of real ones. Raise it to keep only firm contacts; lower it to include
more neighbours.

In [ ]:
adj0 = h.read_adjacency(have[0], adj_dir)
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.hist(np.log10(adj0["contact_voxels"].clip(lower=1)), bins=60, color="#0072B2")
ax.axvline(np.log10(max(MIN_CONTACT, 1)), color="#D55E00")
ax.set_xlabel("log10 contact_voxels")
ax.set_ylabel("number of contacts")
ax.set_title("Contacts in the first stack (line = MIN_CONTACT)", fontsize=9)
fig.tight_layout()
plt.show()
print(f"MIN_CONTACT={MIN_CONTACT} keeps {(adj0['contact_voxels'] >= MIN_CONTACT).mean():.0%} of the contacts in this stack.")

## 3. Average each cell with its neighbours

For every recipe in `NICHE_CONFIGS`, each cell's PCs are mixed with those of the
cells it touches. Cells that touch no other cell (their contacts are smaller than
`MIN_CONTACT`, or their neighbours are not in the table) have no neighbourhood,
and so have the cells of a stack without a contact file: they are counted below
and left out of the niche analysis.

In [ ]:
niche = {}
for nc in NICHE_CONFIGS:
    mat, stat = h.niche_diffuse(
        pcs, emb["stack_name"], emb["label"], adj_dir,
        self_weight=nc["self_weight"], n_iter=nc["n_iter"],
        min_contact=MIN_CONTACT, weight_by_contact=WEIGHT_BY_CONTACT,
    )
    valid = ~np.isnan(mat[:, 0])
    niche[nc["name"]] = {"pcs": mat, "valid": valid, "stats": stat, "cfg": nc}
    no_file = int(stat.loc[stat["note"] != "", "cells"].sum())
    touching_none = int(stat["cells_without_contact"].sum()) - no_file
    print(f"{nc['name']}: {valid.sum():,} of {len(emb):,} cells have a neighbourhood; "
          f"{touching_none:,} touch no cell (contact below MIN_CONTACT, or partner not in the table); "
          f"{no_file:,} are in stacks without a contact file.")

if PRIMARY_CONFIG not in niche:
    raise ValueError(f"PRIMARY_CONFIG '{PRIMARY_CONFIG}' is not one of {list(niche)}")
share_valid = niche[PRIMARY_CONFIG]["valid"].mean()
if niche[PRIMARY_CONFIG]["valid"].sum() < 50:
    raise ValueError("Fewer than 50 cells have a neighbourhood. Check ADJACENCY_DIR, the stack names and MIN_CONTACT.")
if share_valid < 0.8:
    print(f"Warning: only {share_valid:.0%} of the cells have a neighbourhood. Lower MIN_CONTACT, or check that the "
          "contact files belong to this table.")
niche[PRIMARY_CONFIG]["stats"]

## 4. Map and cluster the neighbourhoods

For each recipe: a UMAP map of the averaged values and Leiden clusters at
several resolutions. These are the **niche clusters**. The grid shows every
recipe (rows) at every resolution (columns). More hops give smoother, larger
patches. In "pure" recipes (`self_weight` 0) the cell's own value is not added back at each step.

In [ ]:
for name, nr in niche.items():
    X = nr["pcs"][nr["valid"]]
    if MAX_CELLS_FOR_MAP is not None and len(X) > MAX_CELLS_FOR_MAP:
        nr["map_idx"] = np.sort(np.random.default_rng(RANDOM_SEED).choice(len(X), int(MAX_CELLS_FOR_MAP), replace=False))
    else:
        nr["map_idx"] = np.arange(len(X))
    nr["umap"] = h.run_umap(X[nr["map_idx"]], UMAP_NEIGHBORS, UMAP_MIN_DIST, "euclidean", RANDOM_SEED)
    print(name, end=": ")
    nr["clusters"] = h.leiden_multi(X, NICHE_LEIDEN_RES, k=KNN, seed=RANDOM_SEED)

In [ ]:
fig, axes = plt.subplots(len(niche), len(NICHE_LEIDEN_RES),
                         figsize=(4.2 * len(NICHE_LEIDEN_RES), 3.6 * len(niche)), squeeze=False)
for r, (name, nr) in enumerate(niche.items()):
    for c, res in enumerate(NICHE_LEIDEN_RES):
        m = nr["clusters"][res]
        h.scatter_by(axes[r, c], nr["umap"][:, 0], nr["umap"][:, 1], m[nr["map_idx"]], h.assign_palette(m), s=2,
                     title=f"{name} | resolution {res} ({len(set(m))} niches)", legend=False)
fig.tight_layout()
if SAVE_FIGURES:
    h.save_fig(fig, "niche_umap_grid", fig_dir)
plt.show()

## 5. Which shapes live in which niches?

This table compares the shape clusters (first notebook) with the niche
clusters of the primary recipe. Each value is `log2(observed / expected)`:

- **0**: the shape cluster is in this niche as often as chance would give.
- **+1**: twice as often. **-1**: half as often.
- Values are limited to between -3 and +3.

A shape cluster that is strongly enriched in one niche tends to occur in one
kind of surroundings. A shape cluster spread evenly over all niches does not
depend on its surroundings.

In [ ]:
P = niche[PRIMARY_CONFIG]
valid = P["valid"]
niche_labels = P["clusters"][PRIMARY_NICHE_RES]
emb_valid = emb[valid].reset_index(drop=True)

if SHAPE_COL is not None:
    table = h.enrichment_log2(emb_valid[SHAPE_COL].to_numpy(), niche_labels)
    table.index = [f"S{i}" for i in table.index]
    table.columns = [f"N{i}" for i in table.columns]
    fig = h.plot_enrichment(table, "Niche cluster", "Shape cluster",
                            f"Shape x niche, {PRIMARY_CONFIG}, niche resolution {PRIMARY_NICHE_RES}")
    if SAVE_FIGURES:
        h.save_fig(fig, "shape_by_niche_enrichment", fig_dir)
    plt.show()
    table.to_csv(out_dir / "shape_by_niche_enrichment.csv")
else:
    print("No shape cluster column found; the table is skipped.")

# The same map, coloured two ways.
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
u = P["umap"]
mi = P["map_idx"]
h.scatter_by(axes[0], u[:, 0], u[:, 1], niche_labels[mi], h.assign_palette(niche_labels), title="Coloured by niche cluster")
if SHAPE_COL is not None:
    sc = emb_valid[SHAPE_COL].to_numpy()
    h.scatter_by(axes[1], u[:, 0], u[:, 1], sc[mi], h.assign_palette(sc), title="Coloured by shape cluster")
fig.tight_layout()
if SAVE_FIGURES:
    h.save_fig(fig, "niche_umap_primary", fig_dir)
plt.show()

## 6. Typical cells of each niche

For each niche cluster, the cells whose neighbourhood is closest to the middle
of the cluster. They are written to `representative_cells.csv`. Drawings work
as in the first notebook, when you give `LABELS_TIFF` or `MESHES_DIR`.

In [ ]:
for name, nr in niche.items():
    X = nr["pcs"][nr["valid"]]
    sub = emb[nr["valid"]].reset_index(drop=True)
    for res in NICHE_LEIDEN_RES:
        sub["niche_cluster"] = nr["clusters"][res]
        reps = h.select_representatives(X, sub, "niche_cluster", N_REPS)
        d = out_dir / f"niche_{name}" / f"leiden_res_{float(res)}"
        d.mkdir(parents=True, exist_ok=True)
        reps.to_csv(d / "representative_cells.csv", index=False)
print("Saved typical cells for", len(niche), "recipes.")

sub = emb_valid.copy()
sub["niche_cluster"] = niche_labels
reps_primary = h.select_representatives(P["pcs"][valid], sub, "niche_cluster", N_REPS)
reps_primary.head(10)

In [ ]:
labels_volume = h.read_labels_volume(LABELS_TIFF) if LABELS_TIFF else None
stacks = sorted(emb["stack_name"].unique())
labels_stack = LABELS_STACK_NAME or (stacks[0] if len(stacks) == 1 else None)

if labels_volume is None and MESHES_DIR is None:
    print("No LABELS_TIFF or MESHES_DIR given: drawings skipped.")
else:
    pal = h.assign_palette(niche_labels)
    fig = h.plot_cluster_gallery(reps_primary, pal, labels_volume, labels_stack, VOXEL_SIZE_ZYX, MESHES_DIR)
    if fig is None:
        print("No mesh could be made for the typical cells (check the stack name and the folder).")
    else:
        if SAVE_FIGURES:
            h.save_fig(fig, "typical_cells_per_niche", fig_dir)
        plt.show()

## 7. Save

`niche_embedding.csv` is the shape table plus the niche columns of the primary
recipe: `niche_PC1`..., `niche_umap_1`, `niche_umap_2` and
`niche_leiden_r<resolution>`. Cells without a neighbourhood have empty values, and so do cells left out of a map
(`MAX_CELLS_FOR_MAP`) in the two `niche_umap` columns.
To export another recipe, change `PRIMARY_CONFIG` and run the notebook again.

In [ ]:
res_df = emb.copy()
pc_n = len(pc_cols)
full_pcs = np.full((len(emb), pc_n), np.nan)
full_pcs[valid] = P["pcs"][valid]
for i in range(pc_n):
    res_df[f"niche_PC{i + 1}"] = full_pcs[:, i]
um = np.full((len(emb), 2), np.nan)
um[np.where(valid)[0][P["map_idx"]]] = P["umap"]
res_df["niche_umap_1"], res_df["niche_umap_2"] = um[:, 0], um[:, 1]
for res in NICHE_LEIDEN_RES:
    col = pd.array([pd.NA] * len(emb), dtype="Int64")
    col = pd.Series(col, index=emb.index)
    col[valid] = P["clusters"][res]
    res_df[h.res_name(res, "niche_leiden_r")] = col
res_df.to_csv(out_dir / "niche_embedding.csv", index=False)

h.write_json(out_dir / "niche_analysis_config.json", {
    "shape_table": emb_path.name,
    "n_cells": len(emb),
    "n_cells_with_neighbourhood": int(valid.sum()),
    "n_cells_without_neighbourhood": int((~valid).sum()),
    "max_cells_for_map": MAX_CELLS_FOR_MAP,
    "niche_configs": NICHE_CONFIGS,
    "primary_config": PRIMARY_CONFIG,
    "primary_niche_resolution": PRIMARY_NICHE_RES,
    "shape_column": SHAPE_COL,
    "min_contact": MIN_CONTACT,
    "weight_by_contact": WEIGHT_BY_CONTACT,
    "niche_leiden_resolutions": NICHE_LEIDEN_RES,
    "knn": KNN,
    "umap_neighbors": UMAP_NEIGHBORS,
    "umap_min_dist": UMAP_MIN_DIST,
    "random_seed": RANDOM_SEED,
})
print("Wrote niche_embedding.csv with", res_df.shape[1], "columns.")